# 02 - Prepare the data

**Owner:** Daria  |  **Output:** `data/processed/` (risks_clean.csv, triggers.csv, resolved_risks.csv, demo_R1043.json)

Turns the export into the tables the app and the scoring need:
- every risk with a numeric ACAPS probability and a 0/1 outcome
- one row per trigger (the check-in card shows one line per trigger)
- the resolved risks only (for testing the scoring)
- everything the demo screen needs for R1043 in one small JSON file for Diana

## 0. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# Find the repo folder (the one that contains src/risk_tools.py), wherever this notebook is run from
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "risk_tools.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"Could not find src/risk_tools.py above {Path.cwd()}. Put risk_tools.py in the src/ folder of the repo.")
sys.path.append(str(ROOT / "src"))
from risk_tools import *
print("Repo folder:", ROOT)


Repo folder: c:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth


In [2]:
DATA_PATH   = ROOT / "data" / "risk-radar-export.csv"
OUT_DIR     = ROOT / "data" / "processed"
PROB_SCHEME = "infographic"   # or "pdf" - the two ACAPS documents differ, confirm with Alex
print(PROB_SCHEMES[PROB_SCHEME])

{'very low': 0.1, 'low': 0.3, 'medium': 0.5, 'high': 0.7, 'very high': 0.9}


## 1. Clean table of risks

In [3]:
df = add_outcome(load_risks(DATA_PATH))
df["acaps_p"] = prob_to_number(df["probability"], PROB_SCHEME)
df["n_countries"] = df["country"].fillna("").str.split(",").apply(lambda x: len([c for c in x if c.strip()]))

ds = df["rationale_update"].apply(date_summary)
df["first_date"] = ds.apply(lambda x: x[0])
df["last_date"] = ds.apply(lambda x: x[1])
df["n_dates"] = ds.apply(lambda x: x[2])

print("Risks without a probability rating:", int(df["acaps_p"].isna().sum()))
print("Risks with no status (outcome unknown):", int(df["status"].isna().sum()))
df[["risk_id", "country", "probability", "acaps_p", "risk_level", "status", "outcome"]].head()

Risks without a probability rating: 11
Risks with no status (outcome unknown): 7


,risk_id,country,probability,acaps_p,risk_level,status,outcome
0,R0353,Afghanistan,<NA>,NaN,<NA>,Not materialised,0.0
1,R0990,Afghanistan,Medium,0.5,Medium,Materialised,1.0
2,R1015,Afghanistan,High,0.7,High,Materialised,1.0
3,R1062,Afghanistan,Very High,0.9,High,Active,NaN
4,R1067,Afghanistan,High,0.7,High,Active,NaN


In [8]:
df[["risk_id", "country", "n_countries", "first_date", "last_date", "n_dates"]].head(10)

,risk_id,country,n_countries,first_date,last_date,n_dates
0,R0353,Afghanistan,1,NaT,NaT,0
1,R0990,Afghanistan,1,2025-05-01,2025-05-08,2
2,R1015,Afghanistan,1,2025-02-09,2026-03-31,18
3,R1062,Afghanistan,1,2026-09-27,2026-10-07,3
4,R1067,Afghanistan,1,2026-09-02,2026-10-06,7
5,R1056,Algeria,1,2026-07-01,2026-09-07,11
6,R0371,Bangladesh,1,NaT,NaT,0
7,R1020,Bangladesh,1,NaT,NaT,0
8,R1039,Bangladesh,1,2026-05-01,2026-07-12,10
9,R0510,Burundi,1,NaT,NaT,0


## 2. Triggers: one row per trigger

In [4]:
trig = explode_triggers(df)
print(len(trig), "triggers across", trig["risk_id"].nunique(), "risks")
print("Triggers that could not be parsed:", int((trig["trigger_id"] == "unparsed").sum()))
trig[trig["risk_id"] == "R1043"]

266 triggers across 108 risks
Triggers that could not be parsed: 0


,risk_id,trigger_id,title,description
23,R1043,R1043 - T0618,Expansion of cross-border attacks and insecurity,":Additional drone strikes, shelling, or armed incursions near the Chad–Sudan..."
24,R1043,R1043 - T0619,Attacks on strategic infrastructure in Chad,"Threats by Sudanese actors against Chadian airports, border facilities, mili..."
25,R1043,R1043 - T0620,Renewed proxy mobilisation and cross-border armed group activity,Historical patterns of proxy warfare between Chad and Sudan may re-emerge th...


Risks that have trigger text but no parsed trigger would show up here (this should be empty):

In [5]:
has_text = df.loc[df["triggers"].notna(), "risk_id"]
missing = sorted(set(has_text) - set(trig["risk_id"]))
print(missing or "none")

none


## 3. Resolved risks (for testing the scoring)

In [6]:
resolved = resolved_only(df)
keep = ["risk_id", "risk_title", "country", "probability", "acaps_p", "risk_level", "status", "outcome", "first_date", "last_date"]
resolved = resolved[[c for c in keep if c in resolved.columns]]
print(len(resolved), "resolved risks;", int(resolved["outcome"].sum()), "materialised")
resolved.head()

104 resolved risks; 61 materialised


,risk_id,risk_title,country,probability,acaps_p,risk_level,status,outcome,first_date,last_date
0,R0353,"Taliban attacks on provincial capitals lead to civil war violence levels, re...",Afghanistan,<NA>,NaN,<NA>,Not materialised,0,NaT,NaT
1,R0990,"Unsafe return of Afghan nationals from Pakistan causes food, shelter and liv...",Afghanistan,Medium,0.5,Medium,Materialised,1,2025-05-01,2025-05-08
2,R1015,La Niña in Afghanistan worsens drought and leads to widespread agricultural ...,Afghanistan,High,0.7,High,Materialised,1,2025-02-09,2026-03-31
5,R1056,Resource-related clashes and political tensions in Tindouf camps restrict hu...,Algeria,Medium,0.5,Medium,Materialised,1,2026-07-01,2026-09-07
6,R0371,Heavy rains in late August 2022 cause floods that affect previously affected...,Bangladesh,Medium,0.5,Medium,Not materialised,0,NaT,NaT


Caution: older records resolve over much longer periods than recent ones, so these 104 do not all share a six-month horizon. Keep this in mind, and say so in the pitch.

## 4. Everything the demo screen needs for R1043

In [7]:
row = df[df["risk_id"] == "R1043"].iloc[0]

def val(col):
    return None if col not in row.index or pd.isna(row[col]) else str(row[col])

demo = {
    "risk_id": val("risk_id"), "title": val("risk_title"), "country": val("country"),
    "risk_type": val("risk_type"), "probability": val("probability"), "impact": val("impact"),
    "risk_level": val("risk_level"), "expected_exposure": val("expected_exposure"),
    "status": val("status"), "rationale": val("rationale"),
    "triggers": parse_triggers(row.get("triggers")),
}
print(demo["title"])
print(len(demo["triggers"]), "triggers")

Escalating cross-border insecurity linked to the Sudan conflict increases protection risks and humanitarian needs in eastern Chad
3 triggers


## 5. Save

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "risks_clean.csv", index=False)
trig.to_csv(OUT_DIR / "triggers.csv", index=False)
resolved.to_csv(OUT_DIR / "resolved_risks.csv", index=False)
with open(OUT_DIR / "demo_R1043.json", "w", encoding="utf-8") as f:
    json.dump(demo, f, ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT_DIR.iterdir()))
print("Post in the team chat: processed data is ready.")

['demo_R1043.json', 'resolved_risks.csv', 'risks_clean.csv', 'triggers.csv']
Post in the team chat: processed data is ready.
